# Install openai-agents SDK

In [1]:
!pip install -Uq openai-agents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.5/128.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.4/567.4 kB 7.6 MB/s eta 0:00:00


# Make your Jupyter Notebook capable of running asynchronous functions.

In [3]:
import nest_asyncio
nest_asyncio.apply()

# Run Google Gemini with OPENAI-Agent SDK

In [4]:
import os
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel
from agents.run import RunConfig


API_KEY = os.environ.get("OPENAI_API_KEY")
BASE_URL = os.environ.get("OPENAI_BASE_URL")

# Check if the API key is present; if not, raise an error
if not API_KEY:
    raise ValueError("API_KEY is not set. Please ensure it is defined in your .env file.")

#Reference: https://ai.google.dev/gemini-api/docs/openai
external_client = AsyncOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

model = OpenAIChatCompletionsModel(
    model="gpt-5.4",
    openai_client=external_client
)

config = RunConfig(
    model=model,
    model_provider=external_client,
    tracing_disabled=True
)

# Streaming Text code

In [5]:
import asyncio

from openai.types.responses import ResponseTextDeltaEvent

from agents import Agent, Runner


async def main():
    agent = Agent(
        name="Joker",
        instructions="你叫玲娜贝儿，是我的好朋友。你只说中文。",
        model=model,
    )

    result = Runner.run_streamed(agent, input="Please tell me 5 jokes.", run_config=config)
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)



asyncio.run(main())

当然可以呀，我来给你讲 5 个轻松的小笑话：

1. **老师问小明：**
   “你为什么上课总是睡觉？”
   小明说：“因为老师您说过，梦想还是要有的。”

2. **有一天，苹果对香蕉说：**
   “你能不能把衣服穿好？”
   香蕉说：“不行啊，我一热就忍不住脱皮。”

3. **病人对医生说：**
   “医生，我觉得自己像手机。”
   医生问：“哪里不舒服？”
   病人说：“我总觉得电量不足。”

4. **小鱼问大鱼：**
   “妈妈，为什么我们每天都在水里游？”
   大鱼说：“因为咱们要是上岸，就成干货了。”

5. **老板问员工：**
   “你为什么迟到了？”
   员工说：“因为路上太挤了。”
   老板说：“你不是在家办公吗？”
   员工说：“对啊，卧室去客厅的路上猫躺着呢。”

如果你想，我还可以继续给你讲 5 个更冷的。

# Stream item code

In [6]:
import asyncio
import random

from agents import Agent, ItemHelpers, Runner, function_tool


@function_tool
def how_many_jokes() -> int:
    return random.randint(1, 10)


async def main():
    agent = Agent(
        name="Joker",
        instructions="首先执行 `how_many_jokes` tool, 接下来讲这么多数量的笑话，笑话要与你玲娜贝儿有关。",
        tools=[how_many_jokes],
        model=model,
    )

    result = Runner.run_streamed(
        agent,
        input="Hello",
        run_config=config

    )
    print("=== Run starting ===")
    async for event in result.stream_events():
        # We'll ignore the raw responses event deltas
        if event.type == "raw_response_event":
            continue
        elif event.type == "agent_updated_stream_event":
            print(f"Agent updated: {event.new_agent.name}")
            continue
        elif event.type == "run_item_stream_event":
            if event.item.type == "tool_call_item":
                print("-- Tool was called")
            elif event.item.type == "tool_call_output_item":
                print(f"-- Tool output: {event.item.output}")
            elif event.item.type == "message_output_item":
                print(f"-- Message output:\n {ItemHelpers.text_message_output(event.item)}")
            else:
                pass  # Ignore other event types




try:
  asyncio.run(main())
except:
  pass
print("=== Run complete ===")

=== Run starting ===
Agent updated: Joker
-- Tool was called
-- Tool output: 9
-- Message output:
 你好！给你讲 9 个和玲娜贝儿有关的轻松小笑话：

1. 玲娜贝儿去图书馆，管理员问她借什么书。  
   她说：“《如何优雅地破案》。”  
   管理员问：“你是侦探吗？”  
   她眨眨眼：“不，我只是想先找到下午茶藏在哪儿。”

2. 玲娜贝儿拿着放大镜在草地上找东西。  
   朋友问：“你丢了什么？”  
   她说：“没有，我在找今天的好运气。”  
   朋友问：“找到了吗？”  
   她笑着说：“找到了，你刚刚在跟我说话呀。”

3. 有人问玲娜贝儿为什么总是背着小包。  
   她说：“里面装着秘密。”  
   大家很好奇：“什么秘密？”  
   她小声说：“其实是点心，不能让别人太早发现。”

4. 玲娜贝儿学做数学题。  
   老师问：“一块蛋糕加一块蛋糕等于几块蛋糕？”  
   她立刻回答：“等于下午茶幸福感翻倍！”  
   老师沉默了一下：“……也算一种正确答案。”

5. 玲娜贝儿去花园散步，看见蝴蝶飞来飞去。  
   她认真地拿出笔记本记录。  
   朋友问：“你在研究什么？”  
   她说：“研究怎么飞得这么好看，我走路也不能输。”

6. 玲娜贝儿开会时特别安静。  
   大家以为她在认真思考大计划。  
   结果她抬头说：“我想好了。”  
   大家期待地看着她。  
   她说：“会议结束后吃草莓还是蓝莓甜点？”

7. 玲娜贝儿练习推理。  
   她看着桌上的空盘子说：“我已经知道是谁偷吃了点心。”  
   大家惊讶：“是谁？”  
   她指了指自己嘴角的奶油：“证据有点明显了。”

8. 玲娜贝儿问朋友：“你知道最温柔的天气是什么吗？”  
   朋友猜了半天。  
   她笑着说：“是适合一起散步的天气呀。”  
   朋友说：“这算谜语吗？”  
   她点点头：“算甜甜的那种。”

9. 玲娜贝儿照镜子整理蝴蝶结。  
   朋友问：“你每天都这么认真打扮吗？”  
   她说：“当然呀。”  
   “为什么？”  
   “因为遇见好朋友的时候，心情也要漂漂亮亮的。”

如果你愿意，我还